In [ ]:
import tkinter as tk
from tkinter import ttk
from datetime import datetime, timedelta
import pytz  # Import pytz
import os  # Import the 'os' module
import sys
# import subprocess # Not used, can be removed

def format_datetime(dt):
    dt_copy = dt
    return dt_copy.strftime('%d/%m/%Y')

def format_datetimehour(dt):
    dt_copy = dt
    return dt_copy.strftime('%d/%m/%Y %H:%M:%S')

def resource_path(relative_path):
    """ Get absolute path to resource, works for dev and for PyInstaller """
    try:
        # PyInstaller creates a temp folder and stores path in _MEIPASS
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")

    return os.path.join(base_path, relative_path)

def get_date(entry_widget):
    """Creates a calendar popup and inserts the selected date into the given entry widget."""
    from tkcalendar import Calendar  # Import INSIDE the function

    if not root_exists:  # Prevent interaction if window is closed
        return

    def set_date():
        """Sets the selected date from the calendar to the entry."""
        if not root_exists:  # Check again, inside the nested function
            return
        selected_date = cal.get_date()
        entry_widget.delete(0, tk.END)  # Clear the entry
        entry_widget.insert(0, selected_date)  # Insert the new date
        top.destroy()

    # Create a toplevel window for the calendar
    top = tk.Toplevel(root)

    # Get today's date
    today = datetime.now(current_timezone)

    # Create a Calendar widget
    cal = Calendar(top,
                    font="Arial 10",
                    selectmode='day',
                    year=today.year,
                    month=today.month,
                    day=today.day,
                    date_pattern="yyyy-mm-dd")  # Important: Set the date_pattern

    cal.pack(pady=20)

    # Add a button to confirm the selection
    confirm_button = tk.Button(top, text="OK", command=set_date)
    confirm_button.pack(pady=10)

    # Important: Make the toplevel window transient and grab focus
    top.transient(root)  # Keep the calendar on top of the main window
    top.grab_set()      # Prevent interaction with the main window until closed
    top.wait_window(top)  # Wait for the toplevel to be destroyed

def submit_action():
    global entry_username, combobox_choice, previous_menu
    if not root_exists:
        return

    username = entry_username.get()
    menu_choice = combobox_choice.get()
    print(f"Username: {username}")
    print(f"Menu Choice: {menu_choice}")

    if menu_choice == "1. Input Data Hujan":
        previous_menu = "main"
        show_rainfall_options()
    elif menu_choice == "2. Analisa Pemupukan":
        previous_menu = "main"
        show_estate_options_for_analysis(fertilizer_type)

def show_rainfall_options():
    global label_rainfall_option, back_button, current_menu, button_update_rainfall, button_add_rainfall, previous_menu

    if not root_exists:
        return

    hide_main_widgets()
    current_menu = "rainfall"
    previous_menu = "main"  # CORRECTLY SET previous_menu HERE
    label_rainfall_option = tk.Label(root, text="Choose Rainfall Option:", font=("Arial", 12))
    label_rainfall_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    # --- Update Rainfall Button ---
    button_update_rainfall = tk.Button(root, text="Update the last Daily Rainfall (mm)", command=goto_update_rainfall,
                                        font=("Arial", 10))
    button_update_rainfall.grid(row=1, column=0, padx=10, pady=10, sticky="ew")

    # --- Add Rainfall Button ---
    button_add_rainfall = tk.Button(root, text="Add a new Daily Rainfall (mm)", command=goto_add_rainfall,
                                    font=("Arial", 10))
    button_add_rainfall.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=3, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def goto_update_rainfall():
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Rainfall Option: Update the last Daily Rainfall (mm)")
    # show_rainfall_data_entry()  # WRONG: Go to estate selection FIRST
    show_estate_options()  # CORRECT: Show estate options first
    previous_menu = "rainfall"  # Set previous_menu to rainfall

def goto_add_rainfall():
    global previous_menu, current_time_date
    if not root_exists:
        return
    print(f"Selected Rainfall Option: Add a new Daily Rainfall (mm)")
    # previous_menu = "rainfall"  # Remove: Don't set it here.

    # Check for missing dates and show the missing dates entry if needed
    if check_missing_dates():
        show_missing_dates_entry("rainfall")  # Pass the target menu
    else:
        show_estate_options_for_add_rainfall(current_time_date)


def submit_estate_for_analysis(selected_estate, nama_blok, tanggal_rencana, peilscale, tanggal_terakhir, jenis_terakhir, rencana_jenis):
    global previous_menu, entry_username
    if not root_exists:
        return

    # Get the username
    username = entry_username.get()

    # Placeholder analysis results (replace with your actual logic)
    curah_hujan = "15 mm"  # Example value
    status = "Allowed"  # or "Not Allowed"
    reason = "Sufficient rainfall and soil conditions."  # Example reason
    recommendation = "Proceed with fertilization as planned."  # Example recommendation

    # Display the results
    display_analysis_results(
        selected_estate, nama_blok, tanggal_rencana, peilscale, tanggal_terakhir,
        jenis_terakhir, rencana_jenis, username, curah_hujan, status, reason, recommendation
    )
    # previous_menu = "main"  # No longer going back to main immediately
    # cancel_to_main()

def display_analysis_results(selected_estate, nama_blok, tanggal_rencana, peilscale, tanggal_terakhir,
                                jenis_terakhir, rencana_jenis, username, curah_hujan, status, reason, recommendation):
    global current_menu, label_tanggal_analisa, label_nama_user, label_curah_hujan, \
            label_status, label_reason, label_recommendation, label_selected_estate, \
            label_nama_blok, label_tanggal_rencana, label_peilscale_value, \
            label_tanggal_terakhir_value, label_jenis_terakhir_value, \
            label_rencana_jenis_value, back_to_main_button, reanalyze_button  # Add reanalyze_button

    if not root_exists: return
    hide_estate_widgets()
    current_menu = "analysis_results"

    # --- Display Analysis Results ---
    current_time_input = datetime.now(current_timezone)
    label_tanggal_analisa = tk.Label(root, text=f"Tanggal Analisa: {current_time_input.strftime('%Y-%m-%d %H:%M:%S')}", font=("Arial", 12))
    label_tanggal_analisa.grid(row=0, column=0, padx=10, pady=5, sticky="ew")

    label_nama_user = tk.Label(root, text=f"Nama User: {username}", font=("Arial", 12))
    label_nama_user.grid(row=1, column=0, padx=10, pady=5, sticky="ew")

    label_selected_estate = tk.Label(root, text=f"Selected Estate: {selected_estate}", font=("Arial", 12))
    label_selected_estate.grid(row=2, column=0, padx=10, pady=5, sticky="ew")

    label_nama_blok = tk.Label(root, text=f"Nama Blok: {nama_blok}", font=("Arial", 12))
    label_nama_blok.grid(row=3, column=0, padx=10, pady=5, sticky="ew")

    label_curah_hujan = tk.Label(root, text=f"Curah Hujan: {curah_hujan}", font=("Arial", 12))
    label_curah_hujan.grid(row=4, column=0, padx=10, pady=5, sticky="ew")

    label_peilscale_value = tk.Label(root, text=f"Nilai Peilscale: {peilscale}", font=("Arial", 12))
    label_peilscale_value.grid(row=5, column=0, padx=10, pady=5, sticky="ew")

    label_jenis_terakhir_value = tk.Label(root, text=f"Jenis Pupuk Terakhir: {jenis_terakhir}", font=("Arial", 12))
    label_jenis_terakhir_value.grid(row=6, column=0, padx=10, pady=5, sticky="ew")

    label_tanggal_terakhir_value = tk.Label(root, text=f"Tanggal Pupuk Terakhir: {tanggal_terakhir}", font=("Arial", 12))
    label_tanggal_terakhir_value.grid(row=7, column=0, padx=10, pady=5, sticky="ew")

    label_rencana_jenis_value = tk.Label(root, text=f"Rencana Jenis Pupuk: {rencana_jenis}", font=("Arial", 12))
    label_rencana_jenis_value.grid(row=8, column=0, padx=10, pady=5, sticky="ew")

    label_tanggal_rencana = tk.Label(root, text=f"Tanggal Rencana Pupuk: {tanggal_rencana}", font=("Arial", 12))
    label_tanggal_rencana.grid(row=9, column=0, padx=10, pady=5, sticky="ew")

    label_status = tk.Label(root, text=f"Status: {status}", font=("Arial", 12, "bold"))
    label_status.grid(row=10, column=0, padx=10, pady=5, sticky="ew")

    label_reason = tk.Label(root, text=f"Reason: {reason}", font=("Arial", 12))
    label_reason.grid(row=11, column=0, padx=10, pady=5, sticky="ew")

    label_recommendation = tk.Label(root, text=f"Recommendation: {recommendation}", font=("Arial", 12))
    label_recommendation.grid(row=12, column=0, padx=10, pady=5, sticky="ew")


    # --- Re-analyze Button --- (Row 13)
    reanalyze_button = tk.Button(root, text="Re-analyze", command=go_to_reanalyze, font=("Arial", 10))
    reanalyze_button.grid(row=13, column=0, padx=10, pady=10)

    # --- Back to Main Menu Button --- (Row 14)
    back_to_main_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10))
    back_to_main_button.grid(row=14, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def show_estate_options_for_analysis(fertilizer_type):
    # Declare ALL globals at the TOP of the function
    global label_estate_option, combobox_estate, submit_estate_button, back_button, current_menu, \
            entry_blok, entry_tanggal_rencana_pupuk, entry_peilscale, entry_tanggal_pupuk_terakhir, \
            combobox_jenis_pupuk_terakhir, combobox_rencana_jenis_pupuk, label_blok, label_tanggal_rencana_pupuk, \
            label_peilscale, label_tanggal_pupuk_terakhir, label_jenis_pupuk_terakhir, label_rencana_jenis_pupuk, \
            button_tanggal_rencana_pupuk, button_tanggal_pupuk_terakhir  # Add buttons to global list

    if not root_exists:
        return

    hide_main_widgets()
    current_menu = "estate_analysis"

    # --- Use sticky="ew" on ALL widgets ---
    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=5, sticky="ew")

    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=5, sticky="ew")

    label_blok = tk.Label(root, text="Masukkan Nama Blok:", font=("Arial", 12))
    label_blok.grid(row=2, column=0, padx=10, pady=5, sticky="ew")

    entry_blok = tk.Entry(root, font=("Arial", 10))
    entry_blok.grid(row=3, column=0, padx=10, pady=5, sticky="ew")


    # --- Tanggal Rencana Pupuk ---
    label_tanggal_rencana_pupuk = tk.Label(root, text="Masukkan tanggal rencana pupuk:", font=("Arial", 12))
    label_tanggal_rencana_pupuk.grid(row=4, column=0, padx=10, pady=5, sticky="ew")

    entry_tanggal_rencana_pupuk = tk.Entry(root, font=("Arial", 10))
    entry_tanggal_rencana_pupuk.grid(row=5, column=0, padx=10, pady=5, sticky="ew")

    # Button to open the calendar
    button_tanggal_rencana_pupuk = tk.Button(root, text="Pilih Tanggal", command=lambda: get_date(entry_tanggal_rencana_pupuk))
    button_tanggal_rencana_pupuk.grid(row=5, column=1, padx=5, pady=5) # Place button next to entry


    # --- Tanggal Pupuk Terakhir ---
    label_tanggal_pupuk_terakhir = tk.Label(root, text="Masukkan tanggal pupuk terakhir:", font=("Arial", 12))
    label_tanggal_pupuk_terakhir.grid(row=8, column=0, padx=10, pady=5, sticky="ew")

    entry_tanggal_pupuk_terakhir = tk.Entry(root, font=("Arial", 10))
    entry_tanggal_pupuk_terakhir.grid(row=9, column=0, padx=10, pady=5, sticky="ew")

    # Button to open the calendar
    button_tanggal_pupuk_terakhir = tk.Button(root, text="Pilih Tanggal", command=lambda: get_date(entry_tanggal_pupuk_terakhir))
    button_tanggal_pupuk_terakhir.grid(row=9, column=1, padx=5, pady=5) # Place button next to entry



    label_peilscale = tk.Label(root, text="Masukkan nilai Peilscale:", font=("Arial", 12))
    label_peilscale.grid(row=6, column=0, padx=10, pady=5, sticky="ew")

    entry_peilscale = tk.Entry(root, font=("Arial", 10))
    entry_peilscale.grid(row=7, column=0, padx=10, pady=5, sticky="ew")

    label_jenis_pupuk_terakhir = tk.Label(root, text="Masukkan jenis pupuk terakhir:", font=("Arial", 12))
    label_jenis_pupuk_terakhir.grid(row=10, column=0, padx=10, pady=5, sticky="ew")

    combobox_jenis_pupuk_terakhir = ttk.Combobox(root, values=fertilizer_type, width=30, font=("Arial", 10))
    combobox_jenis_pupuk_terakhir.grid(row=11, column=0, padx=10, pady=5, sticky="ew")

    label_rencana_jenis_pupuk = tk.Label(root, text="Masukkan rencana jenis pupuk:", font=("Arial", 12))
    label_rencana_jenis_pupuk.grid(row=12, column=0, padx=10, pady=5, sticky="ew")

    combobox_rencana_jenis_pupuk = ttk.Combobox(root, values=fertilizer_type, width=30, font=("Arial", 10))
    combobox_rencana_jenis_pupuk.grid(row=13, column=0, padx=10, pady=5, sticky="ew")

    submit_estate_button = tk.Button(root, text="Submit", command=lambda: submit_estate_for_analysis(
        combobox_estate.get(),
        entry_blok.get(),
        entry_tanggal_rencana_pupuk.get(),
        entry_peilscale.get(),
        entry_tanggal_pupuk_terakhir.get(),
        combobox_jenis_pupuk_terakhir.get(),
        combobox_rencana_jenis_pupuk.get()
    ), font=("Arial", 10))
    submit_estate_button.grid(row=14, column=0, padx=10, pady=10)

    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=15, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)
    root.columnconfigure(1, weight=0) # Add this to prevent the button from stretching

def go_to_reanalyze():
    global previous_menu
    if not root_exists:
        return
    hide_analysis_results()
    show_estate_options_for_analysis(fertilizer_type)
    previous_menu = "estate_analysis"  # Correctly set previous_menu

def back_to_main():
    """Hides all widgets and recreates the main menu."""
    global previous_menu
    if not root_exists:
        return
    hide_all_widgets()  # This is the KEY fix: Hide EVERYTHING
    create_main_widgets()
    previous_menu = "main"

def go_back():
    global previous_menu
    if not root_exists:
        print ("not root_exists")
        return
    
    print ("previous_menu", previous_menu)
    if previous_menu == "main":
        cancel_to_main()
    elif previous_menu == "rainfall":
        hide_estate_widgets()
        hide_rainfall_widgets()
        show_rainfall_options()
    elif previous_menu == "estate":
        hide_estate_widgets()
        hide_rainfall_data_entry_widgets()  # Hide the rainfall entry widgets
        show_estate_options()  # Show the estate options
    elif previous_menu == "estate_analysis":
        hide_estate_widgets()
        create_main_widgets()
    # Go back from estate_add_rainfall
    elif previous_menu == "estate_add_rainfall":
        hide_estate_widgets()
        show_rainfall_options()
    elif previous_menu == "analysis_results":
        hide_analysis_results()
        show_estate_options_for_analysis(fertilizer_type)
    #Add missing_date
    elif previous_menu == "missing_date":
        hide_missing_dates_entry()
        create_main_widgets()

def hide_rainfall_data_entry_widgets():
    """Hides the widgets specific to the rainfall data entry screen."""
    if not root_exists:
        return

    try: label_update_rainfall.grid_forget()
    except AttributeError: pass
    try: entry_update_rainfall.grid_forget()
    except AttributeError: pass
    try: submit_update_rainfall_button.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()  # Hide local back button
    except AttributeError: pass
    try: main_menu_button.grid_forget()  # Hide main menu button
    except AttributeError: pass

def hide_analysis_results():
    print("hide_analysis_results")
    if not root_exists: return
    try: label_tanggal_analisa.grid_forget()
    except AttributeError: pass
    try: label_nama_user.grid_forget()
    except AttributeError: pass
    try: label_curah_hujan.grid_forget()
    except AttributeError: pass
    try: label_status.grid_forget()
    except AttributeError: pass
    try: label_reason.grid_forget()
    except AttributeError: pass
    try: label_recommendation.grid_forget()
    except AttributeError: pass
    try: label_selected_estate.grid_forget()
    except AttributeError: pass
    try: label_nama_blok.grid_forget()
    except AttributeError: pass
    try: label_tanggal_rencana.grid_forget()
    except AttributeError: pass
    try: label_peilscale_value.grid_forget()
    except AttributeError: pass
    try: label_tanggal_terakhir_value.grid_forget()
    except AttributeError: pass
    try: label_jenis_terakhir_value.grid_forget()
    except AttributeError: pass
    try: label_rencana_jenis_value.grid_forget()
    except AttributeError: pass
    try: back_to_main_button.grid_forget()
    except AttributeError: pass
    try: reanalyze_button.grid_forget() #hide reanalyze button
    except AttributeError: pass

    if 'current_menu' in globals():
        global current_menu
        current_menu = None

def submit_rainfall_option(selected_rainfall_option):
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Rainfall Option: {selected_rainfall_option}")

    if selected_rainfall_option == "Update the last Daily Rainfall (mm)":
        previous_menu = "rainfall"
        show_estate_options()
    elif selected_rainfall_option == "Add a new Daily Rainfall (mm)":
        # --- Call the new function for adding rainfall ---
        previous_menu = "rainfall"
        show_estate_options_for_add_rainfall()
        # -------------------------------------------------

def show_estate_options_for_add_rainfall(current_time_date):
    global label_estate_option, combobox_estate, submit_estate_add_rainfall_button, back_button, current_menu, entry_daily_rainfall, label_daily_rainfall, main_menu_button, previous_menu
    if not root_exists:
        return

    hide_rainfall_widgets()  # Hide rainfall options
    current_menu = "estate_add_rainfall"  # Differentiate menu
    previous_menu = "rainfall" #add previous menu

    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=10)

    # --- Add Daily Rainfall Input with Date ---
    formatted_date = format_datetime(current_time_date)  # Format the date
    label_daily_rainfall = tk.Label(root, text=f"Masukkan Daily Rainfall (mm) hari ini ({formatted_date}):", font=("Arial", 12))  # Use f-string
    label_daily_rainfall.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    entry_daily_rainfall = tk.Entry(root, font=("Arial", 10))
    entry_daily_rainfall.grid(row=3, column=0, padx=10, pady=10, sticky="ew")
    # ---------------------------------

    # New button for adding rainfall
    submit_estate_add_rainfall_button = tk.Button(root, text="Submit Estate",
                                                    command=lambda: submit_estate_for_add_rainfall(
                                                        combobox_estate.get(), entry_daily_rainfall.get()),
                                                    font=("Arial", 10))
    submit_estate_add_rainfall_button.grid(row=4, column=0, padx=10, pady=10)

    # --- Back Button (goes back one level) ---
    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=5, column=0, padx=10, pady=10)

    # --- Back to Main Menu Button ---
    main_menu_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10))
    main_menu_button.grid(row=6, column=0, padx=10, pady=10)  # Put it on a new row

    root.columnconfigure(0, weight=1)

def submit_estate_for_add_rainfall(selected_estate, daily_rainfall):
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Estate for Add Rainfall: {selected_estate}")
    print(f"Daily Rainfall (mm): {daily_rainfall}")  # Print rainfall
    # Add your logic for handling the new rainfall data here.
    # After processing, you'll likely go back to the main menu:
    previous_menu = "main"
    cancel_to_main()

def show_estate_options():
    global label_estate_option, combobox_estate, submit_estate_button, back_button, current_menu, main_menu_button
    if not root_exists:
        return

    hide_rainfall_widgets()  # Added to remove option buttons
    hide_estate_widgets()  # Added. very important
    hide_rainfall_data_entry_widgets() # VERY IMPORTANT: remove this
    current_menu = "estate"
    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")
    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=10)
    submit_estate_button = tk.Button(root, text="Submit Estate", command=lambda: submit_estate(combobox_estate.get()),
                                        font=("Arial", 10))
    submit_estate_button.grid(row=2, column=0, padx=10, pady=10)
    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))  # Add back button
    back_button.grid(row=3, column=0, padx=10, pady=10)
    main_menu_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10))
    main_menu_button.grid(row=4, column=0, padx=10, pady=10)


    root.columnconfigure(0, weight=1)

def submit_estate(selected_estate):
    global previous_menu
    if not root_exists:
        return
    print(f"Selected Estate: {selected_estate}")
    # previous_menu = "estate"  # Don't set previous_menu here
    show_rainfall_data_entry()  # NOW show rainfall data entry

def show_rainfall_data_entry():
    global previous_menu, entry_update_rainfall, label_update_rainfall, back_button, main_menu_button, submit_update_rainfall_button

    if not root_exists:
        return

    hide_estate_widgets()
    hide_rainfall_widgets()  # Added to remove option buttons
    hide_rainfall_data_entry_widgets() # VERY IMPORTANT: Clear any previous widgets

    previous_menu = "estate"  # Correctly set previous_menu here -- UNCOMMENT THIS LINE

    label_update_rainfall = tk.Label(root, text=f"Masukkan update daily rainfall (mm):", font=("Arial", 12))
    label_update_rainfall.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    entry_update_rainfall = tk.Entry(root, font=("Arial", 10))
    entry_update_rainfall.grid(row=3, column=0, padx=10, pady=10, sticky="ew")

    submit_update_rainfall_button = tk.Button(root, text="Submit Rainfall", command=submit_update_rainfall, font=("Arial", 10))
    submit_update_rainfall_button.grid(row=4, column=0, padx=10, pady=10)

    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=5, column=0, padx=10, pady=10)

    # --- Back to Main Menu Button ---
    main_menu_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10))
    main_menu_button.grid(row=6, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def submit_update_rainfall():
    global previous_menu
    if not root_exists:
        return
    print(f"Rainfall need to be updated (mm): {entry_update_rainfall.get()}")
    previous_menu = "main"  # now go to main menu
    cancel_to_main()

def hide_all_widgets():
    """Hides ALL widgets in the application."""
    if not root_exists:
        return

    for widget in root.winfo_children():  # Iterate through ALL child widgets
        try:
            widget.grid_forget()
        except AttributeError:
            pass  # Just in case, though winfo_children() should only return widgets

def cancel_to_main(): #keep this, but we don't need to call hide functions anymore
    if not root_exists:
        return
    back_to_main() #call back_to_main

def hide_main_widgets():
    if not root_exists: return
    try: label_username.grid_forget()
    except AttributeError: pass
    try: entry_username.grid_forget()
    except AttributeError: pass
    try: button_input_hujan.grid_forget()
    except AttributeError: pass
    try: button_analisa_pemupukan.grid_forget()
    except AttributeError: pass
    try: exit_button.grid_forget()
    except AttributeError: pass

def hide_rainfall_widgets():
    if not root_exists: return
    try: label_rainfall_option.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()
    except AttributeError: pass
    try: submit_estate_add_rainfall_button.grid_forget()  # Also hide this one here
    except AttributeError: pass
    # --- Hide the new buttons ---
    try: button_update_rainfall.grid_forget()
    except AttributeError: pass
    try: button_add_rainfall.grid_forget()
    except AttributeError: pass
    try: label_update_rainfall.grid_forget()
    except AttributeError: pass
    try: entry_update_rainfall.grid_forget()
    except AttributeError: pass
    try: submit_update_rainfall_button.grid_forget()
    except AttributeError: pass
    try: main_menu_button.grid_forget()
    except AttributeError: pass

    global previous_menu
    previous_menu = "rainfall" #change this to rainfall

    if 'current_menu' in globals() :
        global current_menu
        current_menu = None

def hide_estate_widgets():
    global current_menu  # Declare it globally *first*

    if not root_exists: return

    # Use try-except for ALL widget hiding
    try: label_estate_option.grid_forget()
    except AttributeError: pass
    try: combobox_estate.grid_forget()
    except AttributeError: pass
    try: submit_estate_button.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()
    except AttributeError: pass
    try: main_menu_button.grid_forget() #Added main menu button
    except AttributeError: pass
    try: label_blok.grid_forget()
    except AttributeError: pass
    try: entry_blok.grid_forget()
    except AttributeError: pass
    try: label_tanggal_rencana_pupuk.grid_forget()
    except AttributeError: pass
    try: entry_tanggal_rencana_pupuk.grid_forget()
    except AttributeError: pass

    # Only try to hide these if we were in the estate_analysis menu
    if 'current_menu' in globals() and current_menu == "estate_analysis":
        try: button_tanggal_rencana_pupuk.grid_forget() # Hide calendar button
        except AttributeError: pass
        try: label_peilscale.grid_forget()
        except AttributeError: pass
        try: entry_peilscale.grid_forget()
        except AttributeError: pass
        try: label_tanggal_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: entry_tanggal_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: button_tanggal_pupuk_terakhir.grid_forget() # Hide calendar button
        except AttributeError: pass
        try: label_jenis_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: combobox_jenis_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: label_rencana_jenis_pupuk.grid_forget()
        except AttributeError: pass
        try: combobox_rencana_jenis_pupuk.grid_forget()
        except AttributeError: pass

    try: submit_estate_add_rainfall_button.grid_forget()
    except AttributeError: pass
    try: entry_daily_rainfall.grid_forget()
    except AttributeError: pass
    try: label_daily_rainfall.grid_forget()
    except AttributeError: pass


    if 'current_menu' in globals():
        current_menu = None

def create_main_widgets():
    global label_username, entry_username, submit_button, previous_menu, current_menu, back_button, exit_button, button_input_hujan, button_analisa_pemupukan
    if not root_exists:
        return
    root.geometry("500x400")  # reset geometry

    current_menu = "main"
    label_username = tk.Label(root, text="Enter Username:", font=("Arial", 12))
    label_username.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    entry_username = tk.Entry(root, font=("Arial", 10))
    entry_username.grid(row=1, column=0, padx=10, pady=10, sticky="ew")

    # --- Rainfall Input Button ---
    button_input_hujan = tk.Button(
        root, text="1. Input Data Hujan", command=goto_input_hujan, font=("Arial", 12)
    )
    button_input_hujan.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    # --- Fertilizer Analysis Button ---
    button_analisa_pemupukan = tk.Button(
        root,
        text="2. Analisa Pemupukan",
        command=goto_analisa_pemupukan,
        font=("Arial", 12),
    )
    button_analisa_pemupukan.grid(row=3, column=0, padx=10, pady=10, sticky="ew")

    exit_button = tk.Button(root, text="Exit", command=on_closing, font=("Arial", 10))
    exit_button.grid(row=5, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

    previous_menu = None
    back_button = None  # Back button will be re-created as needed

def goto_input_hujan():
    global previous_menu
    if not root_exists: return
    previous_menu = "main"
    print("goto_input_hujan")
    hide_all_widgets() #add this
    
    # Check for missing dates, and if there are, show the entry form
    if check_missing_dates():
        show_missing_dates_entry("rainfall")
    else:
        show_rainfall_options()


def goto_analisa_pemupukan():
    global previous_menu, fertilizer_type
    if not root_exists: return
    previous_menu = "main"
    print("goto_analisa_pemupukan")
    hide_all_widgets() #add this

     # Check for missing dates and show the missing dates entry if needed
    if check_missing_dates():
        show_missing_dates_entry("analysis")  # Pass the target menu
    else:
        show_estate_options_for_analysis(fertilizer_type)

def disable_buttons():
    """Disables all interactive buttons to prevent further events."""
    global back_button, submit_rainfall_button, submit_estate_button, exit_button, submit_estate_add_rainfall_button, button_input_hujan, button_analisa_pemupukan, button_update_rainfall, button_add_rainfall, reanalyze_button, main_menu_button, submit_update_rainfall_button, button_tanggal_rencana_pupuk, button_tanggal_pupuk_terakhir, submit_missing_dates_button  # Add the new buttons

    # Crucial: Check if the root window exists before interacting with ANY widgets.
    if not root_exists:
        return

    # Use try-except blocks for extra safety.
    try:
        if back_button: back_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_rainfall_button: submit_rainfall_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_estate_button: submit_estate_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if exit_button: exit_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_estate_add_rainfall_button: submit_estate_add_rainfall_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_input_hujan: button_input_hujan.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_analisa_pemupukan: button_analisa_pemupukan.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_update_rainfall: button_update_rainfall.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_add_rainfall: button_add_rainfall.config(state="disabled")
    except tk.TclError: pass
    try:
        if reanalyze_button: reanalyze_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if main_menu_button: main_menu_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_update_rainfall_button: submit_update_rainfall_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_tanggal_rencana_pupuk: button_tanggal_rencana_pupuk.config(state="disabled")  # Disable the calendar button
    except tk.TclError: pass
    try:
        if button_tanggal_pupuk_terakhir: button_tanggal_pupuk_terakhir.config(state="disabled")  # Disable the calendar button
    except tk.TclError: pass
    try:
        if submit_missing_dates_button: submit_missing_dates_button.config(state="disabled")
    except tk.TclError: pass
    
def on_closing():
    global root_exists
    root_exists = False  # Set this FIRST
    disable_buttons()  # Disable buttons immediately
    root.destroy()    # Force immediate destruction of the window

# --- Missing Dates Functions ---
def check_missing_dates():
    """Placeholder function to check for missing dates.  Replace with your actual logic."""
    # In a real application, you would check your data source (database, file, etc.)
    # to see if there are any missing dates in the records.
    # For this example, we'll simulate it.
    # return True  # Simulate missing dates
    return False # Simulate NO missing dates

def show_missing_dates_entry(target_menu):
    """Shows a window to enter data for missing dates."""
    global previous_menu, current_menu, entry_missing_date, entry_missing_rainfall, submit_missing_dates_button, label_missing_date, label_missing_rainfall, back_button

    if not root_exists:
        return

    hide_all_widgets()
    current_menu = "missing_date"
    previous_menu = "missing_date"  # Use a consistent name

    label_missing_date = tk.Label(root, text="Enter Missing Date:", font=("Arial", 12))
    label_missing_date.grid(row=0, column=0, padx=10, pady=5, sticky="ew")

    entry_missing_date = tk.Entry(root, font=("Arial", 10))
    entry_missing_date.grid(row=1, column=0, padx=10, pady=5, sticky="ew")
    # Add date picker button
    button_missing_date = tk.Button(root, text="Select Date", command=lambda: get_date(entry_missing_date))
    button_missing_date.grid(row=1, column=1, padx=5, pady=5)


    label_missing_rainfall = tk.Label(root, text="Enter Rainfall (mm):", font=("Arial", 12))
    label_missing_rainfall.grid(row=2, column=0, padx=10, pady=5, sticky="ew")

    entry_missing_rainfall = tk.Entry(root, font=("Arial", 10))
    entry_missing_rainfall.grid(row=3, column=0, padx=10, pady=5, sticky="ew")

    submit_missing_dates_button = tk.Button(root, text="Submit", command=lambda: submit_missing_date_entry(target_menu), font=("Arial", 10))
    submit_missing_dates_button.grid(row=4, column=0, padx=10, pady=10)

    back_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10)) #Consistent naming
    back_button.grid(row=5, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)
    root.columnconfigure(1, weight=0)  # For the date picker button


def submit_missing_date_entry(target_menu):
    """Handles the submission of missing date data."""
    global previous_menu

    if not root_exists:
        return
    
    missing_date = entry_missing_date.get()
    missing_rainfall = entry_missing_rainfall.get()
    
    print(f"Submitted missing date: {missing_date}, Rainfall: {missing_rainfall}")
    
    # Here you would save the entered data to your data storage (e.g., database).

    # After handling the missing date, navigate to the appropriate menu.
    if target_menu == "rainfall":
      show_rainfall_options()
    elif target_menu == "analysis":
      show_estate_options_for_analysis(fertilizer_type)
    previous_menu = "main" # Set to the appropriate previous menu

def hide_missing_dates_entry():
    """Hides the missing dates entry widgets."""
    if not root_exists:
        return

    try: label_missing_date.grid_forget()
    except AttributeError: pass
    try: entry_missing_date.grid_forget()
    except AttributeError: pass
    try: label_missing_rainfall.grid_forget()
    except AttributeError: pass
    try: entry_missing_rainfall.grid_forget()
    except AttributeError: pass
    try: submit_missing_dates_button.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()
    except AttributeError: pass
    try: button_missing_date.grid_forget() #Hide date picker button
    except AttributeError: pass


    if 'current_menu' in globals():
        global current_menu
        current_menu = None

def main_process():
    global root, previous_menu, root_exists, current_menu, \
            submit_button, back_button, submit_rainfall_button, \
            submit_estate_button, exit_button, \
            label_estate_option, combobox_estate, entry_blok, \
            label_tanggal_rencana_pupuk, entry_tanggal_rencana_pupuk, \
            label_peilscale, entry_peilscale, label_tanggal_pupuk_terakhir, \
            entry_tanggal_pupuk_terakhir, label_jenis_pupuk_terakhir, \
            combobox_jenis_pupuk_terakhir, label_rencana_jenis_pupuk, \
            combobox_rencana_jenis_pupuk, label_rainfall_option, \
            combobox_rainfall, submit_estate_add_rainfall_button, \
            entry_daily_rainfall, label_username, entry_username, \
            label_menu_choice, combobox_choice, label_daily_rainfall, label_blok, button_input_hujan, button_analisa_pemupukan, button_update_rainfall, button_add_rainfall, label_tanggal_analisa, label_nama_user, label_curah_hujan, label_status, label_reason, label_recommendation, label_selected_estate, label_nama_blok, label_tanggal_rencana, label_peilscale_value, label_tanggal_terakhir_value, label_jenis_terakhir_value, label_rencana_jenis_value, back_to_main_button, reanalyze_button, current_time_date, main_menu_button, label_update_rainfall, entry_update_rainfall, submit_update_rainfall_button, entry_missing_date, entry_missing_rainfall, submit_missing_dates_button, label_missing_date, label_missing_rainfall, button_missing_date

    root = tk.Tk()
    root.title("Fertilizer Analysis")
    root.state('zoomed')
    previous_menu = None
    root_exists = True
    current_menu = None

    # Initialize ALL widget variables to None
    label_username = None
    entry_username = None
    exit_button = None
    label_rainfall_option = None
    combobox_rainfall = None
    submit_rainfall_button = None  # No longer directly used
    back_button = None
    label_estate_option = None
    combobox_estate = None
    submit_estate_button = None
    entry_blok = None
    label_tanggal_rencana_pupuk = None
    entry_tanggal_rencana_pupuk = None
    label_peilscale = None
    entry_peilscale = None
    label_tanggal_pupuk_terakhir = None
    entry_tanggal_pupuk_terakhir = None
    label_jenis_pupuk_terakhir = None
    combobox_jenis_pupuk_terakhir = None
    label_rencana_jenis_pupuk = None
    combobox_rencana_jenis_pupuk = None
    submit_estate_add_rainfall_button = None
    entry_daily_rainfall = None
    label_daily_rainfall = None
    label_blok = None
    button_input_hujan = None
    button_analisa_pemupukan = None
    button_update_rainfall = None     # New button
    button_add_rainfall = None      # New button
    label_tanggal_analisa = None
    label_nama_user = None
    label_curah_hujan = None
    label_status = None
    label_reason = None
    label_recommendation = None
    label_selected_estate = None
    label_nama_blok = None
    label_tanggal_rencana = None
    label_peilscale_value = None
    label_tanggal_terakhir_value = None
    label_jenis_terakhir_value = None
    label_rencana_jenis_value = None
    back_to_main_button = None
    reanalyze_button = None         # Initialize reanalyze_button
    main_menu_button = None          # Initialize main_menu_button
    label_update_rainfall = None
    entry_update_rainfall = None
    submit_update_rainfall_button = None
    entry_missing_date = None
    entry_missing_rainfall = None
    submit_missing_dates_button = None
    label_missing_date = None
    label_missing_rainfall = None
    button_missing_date = None

    root.protocol("WM_DELETE_WINDOW", on_closing)
    root.columnconfigure(0, weight=1)

    create_main_widgets()
    root.mainloop()
    
if __name__ == "__main__":

    fertilizer_type = ["NPK 13", "NPK 15", "NPK 12", "Dolomite", "Urea", "MOP", "HGFB", "CuSO4", "Zincop Chelated", "Kieserite", "RP", "Kaptan", "TSP"]

    current_timezone = pytz.timezone('Asia/Jakarta')
    date_input = datetime.now(current_timezone)
    current_time_date = datetime.now(current_timezone).date()

    main_process()